# HEX Inference

In [1]:
import openslide
import numpy as np
import pandas as pd
import h5py
import pickle

import os
import torch
import torch.nn as nn

from model_hex_compgat_clpg_cv import HEXContextModel


def parse_fds_active_markers(value, num_markers):
    if value is None or str(value).lower() == "all":
        return list(range(num_markers))
    if str(value).lower() in ["none", ""]:
        return []
    return [int(x.strip()) for x in str(value).split(",") if x.strip()]


def inverse_transform_labels(y_norm, label_mean, label_std, log1p=True):
    y_log = y_norm * (label_std + 1e-6) + label_mean
    if log1p:
        return torch.clamp(torch.expm1(y_log), min=0.0)
    return y_log


def load_single_hex_model(checkpoint_path, device):
    ckpt = torch.load(checkpoint_path, map_location=device)
    args = ckpt.get("args", {})
    num_markers = int(args.get("num_markers", 19))

    model = HEXContextModel(
        in_dim=args.get("in_dim", 1536),
        num_markers=num_markers,
        hidden_dim1=args.get("hex_hidden_dim1", 256),
        hidden_dim2=args.get("hex_hidden_dim2", 128),
        dropout=args.get("hex_dropout", 0.5),

        # vanilla HEX setting
        use_spatial_encoder=args.get("use_spatial_encoder", False),
        gat_mode=args.get("gat_mode", "none"),
        comp_source=args.get("comp_source", "oracle"),
        num_compartments=args.get("num_compartments", 7),
        use_comp_head=args.get("use_comp_head", False),

        # PG-compatible args
        use_protein_graph=args.get("use_protein_graph", False),
        use_prior=args.get("use_prior", False),
        prior_adj=ckpt.get("prior_adj", None),

        # FDS-compatible args
        use_fds=args.get("use_fds", True),
        fds_active_markers=parse_fds_active_markers(
            args.get("fds_active_markers", "all"),
            num_markers,
        ),
        fds_bucket_num=args.get("fds_bucket_num", 50),
        fds_bucket_start=args.get("fds_bucket_start", 0),
        fds_label_min=args.get("fds_label_min", -3.0),
        fds_label_max=args.get("fds_label_max", 3.0),
        fds_momentum=args.get("fds_momentum", 0.9),
        fds_start_smooth=args.get("fds_start_smooth", 10),
        fds_start_update=args.get("fds_start_update", 0),
        fds_kernel=args.get("fds_kernel", "gaussian"),
        fds_ks=args.get("fds_ks", 9),
        fds_sigma=args.get("fds_sigma", 2.0),

        cl_proj_dim=args.get("cl_proj_dim", 64),
    )

    model.load_state_dict(ckpt["model"], strict=True)
    model.to(device)
    model.eval()

    label_mean = ckpt["label_mean"].to(device)
    label_std = ckpt["label_std"].to(device)
    log1p_labels = args.get("log1p_labels", True)

    return model, label_mean, label_std, log1p_labels


class HEX5FoldEnsemble(nn.Module):
    def __init__(
        self,
        cv_root,
        checkpoint_name="best.pt",
        folds=(0, 1, 2, 3, 4),
        device="cuda:0",
    ):
        super().__init__()

        self.device = torch.device(device if torch.cuda.is_available() else "cpu")
        self.members = []

        for fold in folds:
            ckpt_path = os.path.join(cv_root, f"fold{fold}", checkpoint_name)
            model, label_mean, label_std, log1p_labels = load_single_hex_model(
                ckpt_path,
                self.device,
            )
            self.members.append(
                {
                    "model": model,
                    "label_mean": label_mean,
                    "label_std": label_std,
                    "log1p_labels": log1p_labels,
                    "checkpoint": ckpt_path,
                }
            )

    @torch.no_grad()
    def forward(self, h0_optimus_features, return_members=False):
        """
        h0_optimus_features: (1536,), (1, 1536), or (N, 1536)

        return:
            ensemble_mean: (N, 19)
            optionally:
              ensemble_std: (N, 19)
              member_predictions: (num_folds, N, 19)
        """
        x = h0_optimus_features.to(self.device).float()

        if x.ndim == 1:
            x = x.unsqueeze(0)

        dummy_coords = torch.zeros(x.shape[0], 2, device=self.device)

        member_preds_raw = []

        for member in self.members:
            model = member["model"]

            out = model(
                features=x,
                coords=dummy_coords,
                edge_index=None,
                comp_target=None,
                knn_k=8,
                max_edge_distance=None,
                labels=None,
                epoch=0,
            )

            pred_norm = out["pred"]

            pred_raw = inverse_transform_labels(
                pred_norm,
                member["label_mean"],
                member["label_std"],
                log1p=member["log1p_labels"],
            )

            member_preds_raw.append(pred_raw)

        member_preds_raw = torch.stack(member_preds_raw, dim=0)  # (5, N, 19)
        ensemble_mean = member_preds_raw.mean(dim=0)             # (N, 19)
        ensemble_std = member_preds_raw.std(dim=0)               # (N, 19)

        if return_members:
            return ensemble_mean, ensemble_std, member_preds_raw

        return ensemble_mean
    
cv_root = "/mnt/fileserver/lung_pilot/hex_weights"

HEX = HEX5FoldEnsemble(
    cv_root=cv_root,
    checkpoint_name="best.pt",
    folds=(0, 1, 2, 3, 4),
    device="cuda:0",
)


# Inference 

In [2]:
features_optimus = np.load('/mnt/fileserver/lung_pilot/optimus_146/TCGA-05-4245-01A-01-BS1/features_optimus.npy')

device = 'cuda'
features = features_optimus[0,:]
features_tensor = torch.tensor(features).to(device)
HEX = HEX.to(device)
with torch.no_grad():
    output = HEX(features_tensor.unsqueeze(0))

In [3]:
output.shape

torch.Size([1, 19])